In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
from get_light_status import TrafficLightDetector
import torch
import torch.nn as nn
from collections import deque

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Enable memory growth so it doesn't seize the whole card
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

TensorFlow GPU Memory Growth Enabled


In [ ]:
detector = YOLO("yolo26m.pt") 
traffic_light_detector = TrafficLightDetector()

In [ ]:
def get_best_light(lights, img_w, img_h):
    # 1. Define the 'Target Center' (usually the middle of the screen)
    center_x, center_y = img_w / 2, img_h / 2
    
    best_light = None
    min_distance = float('inf')

    for light in lights:
        x1, y1, x2, y2, conf, cls = light
        
        # 2. Calculate the center point of THIS specific bounding box
        l_center_x = (x1 + x2) / 2
        l_center_y = (y1 + y2) / 2
        
        # 3. Calculate distance to image center
        dist = ((l_center_x - center_x)**2 + (l_center_y - center_y)**2)**0.5
        
        # 4. Update if this is the closest one we've seen
        if dist < min_distance:
            min_distance = dist
            best_light = light
            
    return best_light

In [ ]:
# def calculate_advanced_features(light, img_w, img_h, lane_lines=None):
#     x1, y1, x2, y2, conf, cls = light
    
#     # 1. Center of the light
#     l_x = (x1 + x2) / 2
#     l_y = (y1 + y2) / 2
    
#     # 2. Normalized Offset (0 = center, -1 = far left, 1 = far right)
#     offset_x = (l_x - (img_w / 2)) / (img_w / 2)
    
#     # 3. Lane Bias (Simplified: 1 if in center 30% of screen, else lower)
#     # In a real car, this uses the 'Lane Polynomial'
#     lane_bias = 1.0 if abs(offset_x) < 0.15 else 0.2
    
#     # 4. Aspect Ratio (helps Transformer distinguish arrow lights)
#     aspect_ratio = (x2 - x1) / (y2 - y1)
    
#     # Final Vector for Transformer: [offset_x, l_y_norm, lane_bias, aspect_ratio]
#     return torch.tensor([offset_x, l_y / img_h, lane_bias, aspect_ratio])

In [ ]:
def get_transformer_input(frame):
     # 3. Detection (Class 9 is traffic light)
    results = detector(frame, verbose=False, classes=[9])[0]
    
    # We want the MOST confident traffic light detection (class 9 in COCO is traffic light)
    # If YOLO sees 3 lights, we pick the one closest to our lane center
    detections = results.boxes.data # [x1, y1, x2, y2, conf, cls]
    
    if len(detections) > 0:
        # Filter for traffic light class (ID 9)
        lights = detections[detections[:, 5] == 9]
        if len(lights) > 0:
            img_h, img_w = results.orig_shape
            
            best_light = get_best_light(lights,img_w,img_h)
            x1, y1, x2, y2, conf, cls = best_light
            
            # Normalize coordinates (0 to 1)
            
            x_norm = ((x1 + x2) / 2) / img_w
            y_norm = ((y1 + y2) / 2) / img_h
            area = ((x2 - x1) * (y2 - y1)) / (img_w * img_h)
            
            return torch.tensor([x_norm, y_norm, area, conf])
            
    return torch.zeros(4) # Return zeros if no light found

# To fill the '5 frame' sequence for the Transformer:
# sequence = [get_transformer_input(f) for f in last_5_frames]
# transformer_input = torch.stack(sequence).unsqueeze(0) # Shape: (1, 5, 4)

In [ ]:
class TrafficTransformer(nn.Module):
    def __init__(self, input_size=4, num_heads=2):
        super(TrafficTransformer, self).__init__()
        self.embedding = nn.Linear(input_size, 16)
        # The core "Attention" mechanism
        self.attention = nn.MultiheadAttention(embed_dim=16, num_heads=num_heads)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        # Transformer expects (sequence_length, batch, features)
        x = self.embedding(x).permute(1, 0, 2)
        attn_output, attn_weights = self.attention(x, x, x)
        
        # Pool the results and predict relevance
        avg_pool = torch.mean(attn_output, dim=0)
        relevance = torch.sigmoid(self.fc(avg_pool))
        return relevance, attn_weights

model_trans = TrafficTransformer()
# relevance, weights = model_trans(sample_input)
# print(f"Transformer Relevance Score: {relevance.item():.4f}")

In [ ]:
cap = cv2.VideoCapture('video/hongkong2.mp4')


frame_count = 0

OUTPUT_WIDTH = 1280
OUTPUT_HEIGHT = 720

feature_buffer = deque(maxlen=5)
# current_features = torch.zeros(4)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break


    current_features = get_transformer_input(frame)

    # If YOLO missed the light, use the last one we saw to keep the buffer 'warm'
    if torch.all(current_features == 0) and len(feature_buffer) > 0:
        current_features = last_valid_features
    else:
        last_valid_features = current_features
        
    feature_buffer.append(current_features)

    if(len(feature_buffer)==5):
        transformer_input = torch.stack(list(feature_buffer)).unsqueeze(0)

        with torch.no_grad(): # Use no_grad for faster inference
            relevance_score,attn_weights = model_trans(transformer_input)
            # print(f"Current Relevance Score: {relevance_score.item():.4f}")
            score = relevance_score.item()
        
        # --- VISUALIZATION OVERLAY ---
        # Draw a semi-transparent background box for the HUD
        cv2.rectangle(frame, (10, 10), (350, 120), (0, 0, 0), -1)
        
        # Color logic: Green if highly relevant, Red if ignored
        color = (0, 255, 0) if score > 0.7 else (0, 0, 255)
        
        # Display the score
        cv2.putText(frame, f"AI RELEVANCE: {score:.4f}", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        
        # Draw a visual "Relevance Bar"
        bar_width = int(score * 300)
        cv2.rectangle(frame, (20, 70), (20 + bar_width, 90), color, -1)
        cv2.rectangle(frame, (20, 70), (320, 90), (255, 255, 255), 1)

        # Status Text
        status = "LOCKED ON" if score > 0.7 else "SCANNING..."
        cv2.putText(frame, f"STATUS: {status}", (20, 115), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    # Show the result
    display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
    cv2.imshow('Traffic Light Relevance Engine', display_frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break    
    frame_count += 1
    
    
   

cap.release()
cv2.destroyAllWindows()
